In [ ]:
import math
import re
import shutil
import subprocess
from pathlib import Path
from typing import List, Tuple

# ============================================================
# USER CONFIG
# ============================================================

XFOIL_EXE = Path.home() / "Downloads" / "XFOIL6.99" / "xfoil.exe"

OUT_DIR = Path(r"C:\Users\matte\PyCharmMiscProject\aero-surrogate\data\processed\original_polars")

# NACA 00xx (two-digit thickness)
THICKNESS_LIST = [6, 8, 10, 12, 15, 18, 20]

# Reynolds: 50k .. 5M (log grid)
RE_MIN = 5.0e4
RE_MAX = 5.0e6
N_RE = 12

# Mach: 0.2 .. 0.8 (linear grid)
MACH_MIN = 0.2
MACH_MAX = 0.8
N_MACH = 7

# AoA sweep (auto-truncate on first failure)
ALFA_START = -5.0
ALFA_END = 15.0
ALFA_STEP = 0.25

# XFOIL numerics
VISC_ITERS = 200
N_PANELS = 180
TIMEOUT_S = 180

# How many consecutive "no-growth" steps to tolerate before truncating
# (guards against cases where XFOIL doesn't append a row)
MAX_STALL_STEPS = 1

CLEAN_SCRATCH = True

# ============================================================
# GRID HELPERS
# ============================================================

def logspace(a: float, b: float, n: int) -> List[float]:
    la, lb = math.log10(a), math.log10(b)
    if n < 2:
        return [a]
    return [10 ** (la + i * (lb - la) / (n - 1)) for i in range(n)]

def linspace(a: float, b: float, n: int) -> List[float]:
    if n < 2:
        return [a]
    return [a + i * (b - a) / (n - 1) for i in range(n)]

def aoa_list(a1: float, a2: float, da: float) -> List[float]:
    n = int(round((a2 - a1) / da)) + 1
    return [a1 + i * da for i in range(n)]

# ============================================================
# XFOIL HELPERS
# ============================================================

def ensure_xfoil_exists() -> None:
    if not XFOIL_EXE.exists():
        raise FileNotFoundError(f"XFOIL executable not found: {XFOIL_EXE}")

def naca00xx(thickness: int) -> str:
    if not (1 <= thickness <= 99):
        raise ValueError("Thickness must be in [1..99].")
    return f"NACA00{thickness:02d}"

def naca_digits(airfoil: str) -> str:
    s = airfoil.upper().replace("NACA", "")
    if not re.fullmatch(r"\d{4}", s):
        raise ValueError(f"Invalid NACA 4-digit code: {airfoil}")
    return s

def run_xfoil(workdir: Path, script: str) -> subprocess.CompletedProcess:
    return subprocess.run(
        [str(XFOIL_EXE)],
        input=script.encode("utf-8"),
        cwd=str(workdir),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        timeout=TIMEOUT_S,
    )

def dat_to_csv(dat_path: Path, csv_path: Path) -> None:
    lines = dat_path.read_text(errors="ignore").splitlines()

    header_idx = None
    for i, line in enumerate(lines):
        low = line.lower()
        if "alpha" in low and "cl" in low and "cd" in low:
            header_idx = i
            break
    if header_idx is None:
        raise RuntimeError(f"Could not find polar header in {dat_path}")

    header = lines[header_idx].split()
    rows = []
    for line in lines[header_idx + 1:]:
        if not line.strip():
            continue
        if line.strip()[0] not in "-+.0123456789":
            continue
        cols = line.split()
        # keep only as many columns as header has
        rows.append(",".join(cols[: len(header)]))

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    csv_path.write_text(",".join(header) + "\n" + "\n".join(rows) + "\n")

# ============================================================
# AUTO-TRUNCATING POLAR GENERATION
# ============================================================

def build_truncating_polar_script(
    naca_4: str,
    re_val: float,
    mach_val: float,
    polar_dat_name: str,
    alfas: List[float],
) -> str:
    """
    Writes a single XFOIL script that:
      - loads NACA airfoil
      - sets paneling
      - sets OPER/VISC/MACH/ITER
      - opens PACC, then runs ALFA commands sequentially
      - closes PACC and quits

    Truncation is done externally by stopping further ALFA calls (see run_truncating_polar()).
    However, we still generate scripts in chunks, because XFOIL won't tell us programmatically if a specific ALFA failed.
    """
    # Not used directly; we do chunked execution below.
    raise NotImplementedError


def run_truncating_polar(
    workdir: Path,
    naca_4: str,
    re_val: float,
    mach_val: float,
    alfas: List[float],
    polar_dat_name: str,
) -> Tuple[int, bool]:
    """
    Runs XFOIL incrementally to detect failure and truncate.
    Approach:
      - Start XFOIL once per AoA step is costly but gives a clear "did it append?" signal.
      - For dataset generation reliability, we do one AoA per run while reusing the same polar file.
      - Stop when the polar file stops growing (or missing) for MAX_STALL_STEPS consecutive alfas.

    Returns: (n_points_written, truncated)
    """
    polar_path = workdir / polar_dat_name
    n_written = 0
    stalled = 0

    # Create a stable baseline by writing the airfoil + OPER setup once and then appending points.
    # We re-run XFOIL each step but always:
    #   - load the airfoil
    #   - set operating conditions
    #   - PACC to same file (append)
    #   - run ALFA alpha
    # This is robust even if XFOIL crashes mid-way.
    for alpha in alfas:
        size_before = polar_path.stat().st_size if polar_path.exists() else 0

        script = "\n".join([
            f"NACA {naca_4}",
            "PPAR",
            f"N {N_PANELS}",
            "",
            "",
            "OPER",
            f"VISC {re_val:.6g}",
            f"MACH {mach_val:.4f}",
            f"ITER {VISC_ITERS}",
            "PACC",
            polar_dat_name,
            "",
            f"ALFA {alpha:.3f}",
            "PACC",
            "",
            "QUIT",
            ""
        ])

        try:
            run_xfoil(workdir, script)
        except subprocess.TimeoutExpired:
            # treat as failure at this alpha
            stalled += 1
            if stalled > MAX_STALL_STEPS:
                return n_written, True
            continue

        size_after = polar_path.stat().st_size if polar_path.exists() else 0

        # Heuristic: if file didn't grow, assume failure at/after stall
        if size_after <= size_before:
            stalled += 1
            if stalled > MAX_STALL_STEPS:
                return n_written, True
        else:
            stalled = 0
            n_written += 1

    return n_written, False

# ============================================================
# MAIN
# ============================================================

def main() -> None:
    ensure_xfoil_exists()
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    reynolds_list = logspace(RE_MIN, RE_MAX, N_RE)
    mach_list = linspace(MACH_MIN, MACH_MAX, N_MACH)
    alfas = aoa_list(ALFA_START, ALFA_END, ALFA_STEP)

    base_work = OUT_DIR / "_xfoil_work"
    base_work.mkdir(exist_ok=True)

    for t in THICKNESS_LIST:
        airfoil = naca00xx(t)
        naca_4 = naca_digits(airfoil)

        for re_val in reynolds_list:
            for mach_val in mach_list:
                tag = f"{airfoil}_Re{re_val:.2e}_M{mach_val:.2f}"
                dat_name = f"{tag}.dat"
                csv_path = OUT_DIR / f"{tag}.csv"

                workdir = base_work / tag
                workdir.mkdir(parents=True, exist_ok=True)

                try:
                    n_pts, truncated = run_truncating_polar(
                        workdir=workdir,
                        naca_4=naca_4,
                        re_val=re_val,
                        mach_val=mach_val,
                        alfas=alfas,
                        polar_dat_name=dat_name,
                    )

                    dat_path = workdir / dat_name
                    if not dat_path.exists() or dat_path.stat().st_size < 200 or n_pts < 3:
                        print(f"[SKIP] {tag} (no usable polar; n_pts={n_pts})")
                        continue

                    dat_to_csv(dat_path, csv_path)
                    if truncated:
                        print(f"[OK/TRUNC] {tag} (points={n_pts})")
                    else:
                        print(f"[OK] {tag} (points={n_pts})")

                finally:
                    if CLEAN_SCRATCH:
                        shutil.rmtree(workdir, ignore_errors=True)

    if CLEAN_SCRATCH:
        shutil.rmtree(base_work, ignore_errors=True)

    print("\nDone: polars generated with auto-truncation on convergence break.")


if __name__ == "__main__":
    main()

[OK/TRUNC] NACA0006_Re5.00e+04_M0.20 (points=46)
[OK/TRUNC] NACA0006_Re5.00e+04_M0.30 (points=47)
[OK/TRUNC] NACA0006_Re5.00e+04_M0.40 (points=68)
[SKIP] NACA0006_Re5.00e+04_M0.50 (no usable polar; n_pts=0)
[OK/TRUNC] NACA0006_Re5.00e+04_M0.60 (points=38)
[OK/TRUNC] NACA0006_Re5.00e+04_M0.70 (points=7)
[OK/TRUNC] NACA0006_Re5.00e+04_M0.80 (points=39)
[SKIP] NACA0006_Re7.60e+04_M0.20 (no usable polar; n_pts=1)
[SKIP] NACA0006_Re7.60e+04_M0.30 (no usable polar; n_pts=1)
[SKIP] NACA0006_Re7.60e+04_M0.40 (no usable polar; n_pts=1)
[SKIP] NACA0006_Re7.60e+04_M0.50 (no usable polar; n_pts=1)
[SKIP] NACA0006_Re7.60e+04_M0.60 (no usable polar; n_pts=1)
[SKIP] NACA0006_Re7.60e+04_M0.70 (no usable polar; n_pts=1)
[SKIP] NACA0006_Re7.60e+04_M0.80 (no usable polar; n_pts=1)
[SKIP] NACA0006_Re1.16e+05_M0.20 (no usable polar; n_pts=1)
[SKIP] NACA0006_Re1.16e+05_M0.30 (no usable polar; n_pts=1)
[SKIP] NACA0006_Re1.16e+05_M0.40 (no usable polar; n_pts=1)
[SKIP] NACA0006_Re1.16e+05_M0.50 (no usable pol